In [1]:
import os, json
from dotenv import load_dotenv
import subprocess, time
from mcp.client.streamable_http import streamable_http_client
from contextlib import AsyncExitStack
from mcp import ClientSession


import textwrap

import truststore
truststore.inject_into_ssl()

def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [2]:
import os                                                 
# Clear proxy vars for this process (the VPN set them system-wide).                          
for v in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY",                                          
        "http_proxy", "https_proxy", "all_proxy"):                                         
    os.environ.pop(v, None)                                                                  
# Belt-and-braces: tell anything that still reads env to bypass loopback.                    
os.environ["NO_PROXY"] = "127.0.0.1,localhost,::1"                                           
os.environ["no_proxy"] = "127.0.0.1,localhost,::1" 

In [3]:
class MCPHttpHost:
    def __init__(self):
        self.session_by_tool = {}
        self.openai_tools = []
        self._stack = AsyncExitStack()

    async def add_server(self, url):
        # streamablehttp_client yields (read, write, get_session_id)
        read, write, _ = await self._stack.enter_async_context(
            streamable_http_client(url)
        )
        session = await self._stack.enter_async_context(ClientSession(read, write))
        await session.initialize()

        listed = await session.list_tools()
        for t in listed.tools:
            self.session_by_tool[t.name] = session
            self.openai_tools.append({
                "type": "function",
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            })
        print(f"  connected to {url} — tools: {[t.name for t in listed.tools]}")

    async def call(self, tool_name, args):
        session = self.session_by_tool[tool_name]
        result = await session.call_tool(tool_name, args)
        return "\n".join(c.text for c in result.content if hasattr(c, "text"))

    async def close(self):
        await self._stack.aclose()



In [4]:
import json
from openai import OpenAI

openai_client = OpenAI()
MODEL = "gpt-5-nano"


async def chat(host, user_message, verbose=True):
    """Run one user message through an MCP-backed tool-use loop."""
    input_items = [{"role": "user", "content": user_message}]

    while True:
        response = openai_client.responses.create(
            instructions="You must use the provided tools for all operations if possible. Do not compute anything yourself, even trivial values. Chain tools when needed.",
            model=MODEL,
            input=input_items,
            tools=host.openai_tools,
        )

        # Collect any tool calls the model emitted this turn
        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            # No more tool calls — the model is done; return its text reply
            print("No more tool calls — the model is done; returning its text reply.")
            return response.output_text

        # For each tool call: execute via the host and append result
        for tc in tool_calls:
            args = json.loads(tc.arguments)
            if verbose:
                print(f"🔧 {tc.name}({args})")

            result = await host.call(tc.name, args)

            if verbose:
                print(f"   → {result}")

            # Echo the function_call back, then supply its output
            input_items.append({
                "type": "function_call",
                "call_id": tc.call_id,
                "name": tc.name,
                "arguments": tc.arguments,
            })
            input_items.append({
                "type": "function_call_output",
                "call_id": tc.call_id,
                "output": result,
            })


In [5]:
MATH_URL = "http://127.0.0.1:8001/mcp/"
TEXT_URL = "http://127.0.0.1:8002/mcp/"



# --- 3) Run the same three tests, now over HTTP ---------------------------
http_host = MCPHttpHost()
try:
    print("\nConnecting via HTTP…")
    await http_host.add_server(MATH_URL)
    await http_host.add_server(TEXT_URL)

    for q in [
        "What is 13 times 29, then raised to the power of 2?",
        "Reverse the string 'model context protocol' and then shout it.",
        "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
        "and then multiply that count by 13.5.",
    ]:
        print(f"\n❓ {q}")
        answer = await chat(http_host, q)
        print(f"💬 {answer}")
finally:
    # --- 4) Tear down -----------------------------------------------------
    await http_host.close()
    print("\nServers stopped, clients closed.")


Connecting via HTTP…
  connected to http://127.0.0.1:8001/mcp/ — tools: ['add', 'multiply', 'power', 'modulo']
  connected to http://127.0.0.1:8002/mcp/ — tools: ['uppercase', 'word_count', 'reverse']

❓ What is 13 times 29, then raised to the power of 2?
🔧 multiply({'a': 13, 'b': 29})
   → 377.0
🔧 power({'base': 377.0, 'exponent': 2})
   → 142129.0
No more tool calls — the model is done; returning its text reply.
💬 142129.0

❓ Reverse the string 'model context protocol' and then shout it.
🔧 reverse({'text': 'model context protocol'})
   → locotorp txetnoc ledom
🔧 uppercase({'text': 'locotorp txetnoc ledom'})
   → LOCOTORP TXETNOC LEDOM
No more tool calls — the model is done; returning its text reply.
💬 LOCOTORP TXETNOC LEDOM

❓ Count the words in the sentence 'MCP makes tools discoverable at runtime' and then multiply that count by 13.5.
🔧 word_count({'text': 'MCP makes tools discoverable at runtime'})
   → 6
🔧 multiply({'a': 6, 'b': 13.5})
   → 81.0
No more tool calls — the model is

# Agents SDK

In [6]:
from agents import Agent, Runner                                                             
from agents.mcp import MCPServerStreamableHttp                                               
                                                                                            
MATH_URL = "http://127.0.0.1:8001/mcp/"                                                      
TEXT_URL = "http://127.0.0.1:8002/mcp/"


async def agents_sdk_http_demo():
    # Note: `params={"url": ...}` — no command, no args. The SDK just opens                  
    # a streamable-http session to the URL. Servers are fully independent.                   
    math_mcp = MCPServerStreamableHttp(                                                      
        params={"url": MATH_URL},                                                            
        cache_tools_list=True,                                                               
    )                                                                                        
    text_mcp = MCPServerStreamableHttp(                                                    
        params={"url": TEXT_URL},
        cache_tools_list=True,                                                               
    )
                                                                                            
    # `async with` still manages the CLIENT session lifecycle,                               
    # but the server processes keep running after this block exits.
    async with math_mcp, text_mcp:                                                           
        agent = Agent(                                                                     
            name="MCP Demo Assistant",                                                       
            model=MODEL,                                                                   
            instructions=(                                                                   
                "You have access to math and text tools over MCP. "
                "Use them for every operation — do not compute anything yourself, "          
                "even trivial values. Chain tools when needed."                            
            ),                                                                               
            mcp_servers=[math_mcp, text_mcp],
        )                                                                                    
                                                                                            
        questions = [
            "What is 13 times 29, then raised to the power of 2?",
            "Reverse the string 'model context protocol' and then shout it.",                
            "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
            "and then multiply that count by 13.5.",      
            "Find remainder of 123456789 divided by 97, then reverse that number and raise it to the power of 3.",                                   
        ]                                                                                    

        for i, q in enumerate(questions, 1):                                                 
            print(f"\n────────── Test {i} ──────────")                                     
            print(f"❓ {q}")
            result = await Runner.run(starting_agent=agent, input=q)                         
            print(f"💬 {result.final_output}")
                                                                                            
                                                                                            
await agents_sdk_http_demo()                                    


────────── Test 1 ──────────
❓ What is 13 times 29, then raised to the power of 2?
💬 13 × 29 = 377, and 377^2 = 142129.

Answer: 142129

────────── Test 2 ──────────
❓ Reverse the string 'model context protocol' and then shout it.
💬 LOCOTORP TXETNOC LEDOM

────────── Test 3 ──────────
❓ Count the words in the sentence 'MCP makes tools discoverable at runtime' and then multiply that count by 13.5.
💬 - Word count: 6
- 6 × 13.5 = 81.0

Answer: 81.0

────────── Test 4 ──────────
❓ Find remainder of 123456789 divided by 97, then reverse that number and raise it to the power of 3.
💬 - Remainder of 123456789 divided by 97: 39
- Reverse the remainder: 93
- 93 cubed: 93^3 = 804357

Final result: 804357
